# Tumour/Alveolar Epithelial Compartment: FDG Layout and PAGA Trajectory

Subsets the epithelial compartment to a defined donor cohort, clusters using
the scVI (HVG) latent representation, computes a force-directed graph (FDG)
layout with PAGA connectivity, assigns cell states from Leiden clusters, and
summarizes cell-state composition across donors/treatment response groups.

In [ ]:
# !pip install --quiet scanpy leidenalg igraph louvain fa2-modified

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
sc.settings.figdir = "/data/h5ad/tumour/pulmonary_epi_Lin/"

## Config

In [ ]:
input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine.h5ad"
output_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06.h5ad"

# Subset donors from treatment naive pair normal adjacent, on-treatment with gefitinib and tki resistance with first line only# 
donor_ids = [
    "TN001", "TN002", "TN003", "TN004", "TN005", "TN006", "TN007", "TN008", "TN009",
    "P009", "P012", "P001", "P003", "P005", "P011", "P002", "P006", "P008", "P013",
    "R001", "R003", "R006", "R010", "R009", "R005", "R007",
]

# marker gene panel for exploratory dotplot, grouped by treatment response (recist_v3)
marker_genes = {
    "Lipid": {"ABCG1", "CD5L"},
    "Comb": {"SIRPA", "CD47", "CSF1R"},
    "Receptor_costim": {
        "CD40", "CD274", "PDCD1", "CD80", "CD86", "CCR2", "IL4R",
        "CCR5", "IL12A", "MARCO", "ARG1", "CXCR4", "HAVCR2", "SIGLEC10", "IDO1", "LILRB1", "LILRB2",
    },
    "Chemokine_Cytokine": {
        "CCL5", "CCL20", "IL6", "TNF", "IL1B", "NLRP3", "CCL22", "EGF", "IL10",
        "TGFB1", "CCL3", "VEGFA", "MMP9", "EPHA4", "CAMP",
    },
    "TLR_STING": {"TLR7", "TLR9", "TLR8", "TMEM173", "STAT1", "STAT3"},
    "This_study": {"GPNMB", "CD276"},
}

# clustering parameters
latent_rep = "X_scVI_n_latent_hvg_10"
n_neighbors = 40
leiden_resolution = 0.6
leiden_key = "leiden_scvi_hvg_10_n40"

# colors for broad cell-type annotation
cell_type_colors = {
    "Malignant": "#023fa5",
    "Non-malignant epithelial": "#84bcf0",
    "Transitional": "#63ffac",
    "Alveolar cell type I": "#7d87b9",
    "Alveolar cell type II": "#bec1d4",
    "Club cell": "#4a6fe3",
    "Goblet cell": "#8e063b",
}

# maps Leiden cluster numbers -> cell-state labels, and colors per state
cell_state_map = {str(i): f"cl_{i}" for i in range(12)}
cell_state_colors = {
    "cl_0": "#F6991C", "cl_1": "#D03833", "cl_2": "#03a8ff", "cl_3": "#32145E",
    "cl_4": "#44AD9C", "cl_5": "#bb7784", "cl_6": "#8e063b", "cl_7": "#4a6fe3",
    "cl_8": "#c6dec7", "cl_9": "#8dd593", "cl_10": "#d33f6a", "cl_11": "#E6AFB9",
}

# donor display order for composition barplots (adjacent-normal samples first)
donor_order = [
    "TN008_AN", "TN004_AN", "TN006_AN", "TN007_AN", "TN002_AN", "TN001_AN", "TN005_AN", "TN003_AN", "TN009_AN",
    "TN008", "TN004", "TN006", "TN007", "TN002", "TN001", "TN005", "TN003", "TN009",
    "P003", "P002", "P008", "P006", "P012", "P005", "P013", "P001", "P009", "P011",
    "R006", "R003", "R001", "R010", "R009", "R007", "R005",
]
cell_type_order = [
    "Alveolar cell type I", "Alveolar cell type II", "Transitional", "Club cell",
    "Goblet cell", "Non-malignant epithelial", "Malignant",
]

# FDG groups to highlight individually: (obs_column, [values to highlight])
fdg_highlight_groups = [
    ("cell_state", ["cl_0"]),
    ("recist_v3", ["3_on_treatment", "1_treatment_naive"]),
    ("cell_type_new", ["Malignant"]),
    ("cell_type_new", ["Alveolar cell type I", "Alveolar cell type II", "Goblet cell"]),
]

## Load and preprocess data

In [ ]:
adata = sc.read(input_path)
adata = adata[adata.obs["ID"].isin(donor_ids)].copy()

adata.layers["raw_counts"] = adata.X.copy()  # preserve raw counts before normalizing
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata

## Marker gene dotplot (by treatment response)

In [ ]:
axes_dict = sc.pl.dotplot(
    adata, marker_genes, groupby="recist_v3", use_raw=False,
    standard_scale="var", cmap="Blues", figsize=(14, 2),
    show=False, swap_axes=False, dendrogram=False,
)
for ax in axes_dict.values():
    for label in ax.get_xticklabels():
        label.set_style("italic")
plt.show()

## Clustering using scVI (HVG) latent representation

In [ ]:
sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=latent_rep)
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added=leiden_key, resolution=leiden_resolution)
sc.pl.umap(adata, color=[leiden_key, "ID_2"])

## Force-directed graph (FDG) layout and PAGA

In [ ]:
adata.uns["cell_type_new_colors"] = cell_type_colors

sc.tl.draw_graph(adata)
sc.pl.draw_graph(
    adata, color=leiden_key, size=20, legend_loc="on data",
    save="_all_FA_FDG_res06.pdf",
)
sc.pl.draw_graph(
    adata, color="cell_type_new", legend_loc="on data", size=30,
    palette=cell_type_colors, save="_all_FA_FDG_cell_type_new_res06.pdf",
)

In [ ]:
sc.tl.paga(adata, groups=leiden_key)
sc.pl.paga(adata, color=[leiden_key], save="_all_1G2G3G_leiden_scvi_hvg_10_paga_res06.pdf")
sc.pl.paga(adata, color=["cell_type_new"], save="_all_1G2G3G_cell_type_paga_res06.pdf")

sc.pl.paga_compare(
    adata, threshold=0.1, title="", right_margin=0.2, size=20, edge_width_scale=0.25,
    legend_fontsize=12, fontsize=12, frameon=False, edges=True,
    save="_all_1G2G3G_paga_compare_res06.pdf",
)

## Assign cell states from Leiden clusters - only for visualisation not for intepretation / see Hotspot

In [ ]:
adata.obs["cell_state"] = (
    adata.obs[leiden_key].map(lambda x: cell_state_map.get(x, x)).astype("category")
)
adata.uns["cell_state_colors"] = cell_state_colors

sc.pl.draw_graph(
    adata, color=["cell_state"], size=30, palette=cell_state_colors,
    save="_all_1G2G3G_FDG_PAGA_res06.pdf",
)

## Highlight specific groups on the FDG layout

For each `(obs_column, values)` pair in `fdg_highlight_groups`, highlight the given values.

In [ ]:
for obs_col, values in fdg_highlight_groups:
    sc.pl.draw_graph(adata, color=[obs_col], groups=values, size=30)

## Save annotated output

In [ ]:
adata.X = adata.layers["raw_counts"].copy()
if "log1p" in adata.uns:
    del adata.uns["log1p"]

adata.write(output_path)
adata